# Modified Jaynes–Cummings model

Cleaned and organized for reproducible execution from the repository root.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import time
from qutip import *
from qutip.ipynbtools import plot_animation
from matplotlib import animation
from IPython.display import HTML

In [ ]:
def jc_integrate(N, wc, wa, g, kappa, gamma, psi0, use_rwa, tlist):

    # Hamiltonian
    idc = qeye(N)
    ida = qeye(2)

    a  = tensor(destroy(N), ida)
    sm = tensor(idc, destroy(2))

    if use_rwa: 
        # use the rotating wave approxiation
        H = wc * a.dag() * a + wa * sm.dag() * sm + g * (a.dag() * sm + a * sm.dag())
    else:
        H = wc * a.dag() * a + wa * sm.dag() * sm + g * (a.dag() + a) * (sm + sm.dag())
        
    # collapse operators
    c_op_list = []

    n_th_a = 0.0 # zero temperature

    rate = kappa * (1 + n_th_a)
    if rate > 0.0:
        c_op_list.append(np.sqrt(rate) * a)

    rate = kappa * n_th_a
    if rate > 0.0:
        c_op_list.append(np.sqrt(rate) * a.dag())

    rate = gamma
    if rate > 0.0:
        c_op_list.append(np.sqrt(rate) * sm)

    # evolve and calculate return state vectors
    result = mesolve(H, psi0, tlist, c_op_list, [])  

    return result

In [ ]:
# parameters
wc = 1.0 * 2 * np.pi   # cavity frequency
wa = 1.0 * 2 * np.pi   # atom frequency
g  = 0.05 * 2 * np.pi  # coupling strength
kappa = 0.05        # cavity dissipation rate
gamma = 0.15        # atom dissipation rate
N = 10              # number of cavity fock states

use_rwa = True

# initial state
#psi0 = tensor(basis(N,0), basis(2,1))  # start with an excited atom 
#psi0 = tensor(coherent(N,1.5), basis(2,0))  # or a coherent state the in cavity
psi0 = tensor((coherent(N,2.0)+coherent(N,-2.0)).unit(), basis(2,0))  # or a superposition of coherent states

tlist = np.linspace(0, 30, 150)

In [ ]:
result = jc_integrate(N, wc, wa, g, kappa, gamma, psi0, use_rwa, tlist)

In [ ]:
def create_html_animation_with_prob(result, xvec, X, Y, tlist, interval=200):
    """
    Create a HTML5 animation of the Wigner surface and atomic excitation probability over time.
    """
    # Setup figure with 3D and 2D subplots
    fig = plt.figure(figsize=(12, 6))
    gs = fig.add_gridspec(1, 2, width_ratios=[3, 1])
    ax3d = fig.add_subplot(gs[0, 0], projection='3d', azim=-107, elev=49)
    ax2d = fig.add_subplot(gs[0, 1])

    # Identify system dimensions and sigma_minus operator
    dims = result.states[0].dims[0]
    N = dims[0]
    sm = tensor(qeye(N), destroy(2))

    # Precompute atomic excitation probabilities
    ex_list = [expect(sm.dag() * sm, psi) for psi in result.states]

    # Initial Wigner plot
    rho0 = ptrace(result.states[0], 0)
    W0 = wigner(rho0, xvec, xvec)
    surf = ax3d.plot_surface(
        X, Y, W0, rstride=1, cstride=1,
        cmap=plt.cm.jet, alpha=1.0,
        linewidth=0.05, vmax=0.25, vmin=-0.25
    )
    ax3d.set_xlim(-5, 5)
    ax3d.set_ylim(-5, 5)
    ax3d.set_zlim(-0.25, 0.25)
    fig.colorbar(surf, ax=ax3d, shrink=0.65, aspect=20)

    # Initial probability plot
    ax2d.plot(tlist, ex_list)
    marker, = ax2d.plot(tlist[0], ex_list[0], 'o')
    ax2d.set_xlabel('Time')
    ax2d.set_ylabel('Excitation Probability')
    ax2d.set_ylim(0, 1)

    def update(frame):
        # Update Wigner surface
        ax3d.cla()
        rho = ptrace(result.states[frame], 0)
        W = wigner(rho, xvec, xvec)
        surf = ax3d.plot_surface(
            X, Y, W, rstride=1, cstride=1,
            cmap=plt.cm.jet, alpha=1.0,
            linewidth=0.05, vmax=0.25, vmin=-0.25
        )
        ax3d.set_xlim(-5, 5)
        ax3d.set_ylim(-5, 5)
        ax3d.set_zlim(-0.25, 0.25)

        # Update marker on probability plot
        marker.set_data([tlist[frame]], [ex_list[frame]])
        return [surf, marker]

    anim = animation.FuncAnimation(
        fig, update, frames=len(result.states),
        interval=interval, blit=False
    )
    plt.close(fig)
    return HTML(anim.to_jshtml())

# Usage:
anim_html = create_html_animation_with_prob(result, xvec, X, Y, tlist)
anim_html  # Renders the combined animation
